# Bronze Layer Ingestion

## Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from urllib.parse import urljoin, unquote
import requests
import re
import os
import hashlib

## Setting Paths to Catalog, Schemas and Volume

In [0]:
CATALOG = "rf_assessment"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

VOLUME = "rf_assessment_raw"

VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{VOLUME}"

SOURCE_PAGE = "https://ntdc.gov.pk/merit-order"

TARGET_YEAR = 2026
TARGET_MONTHS = {5, 6, 7, 8}

print("Volume:", VOLUME_PATH)

## HTTP Session with Retry Strategy

In [0]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()

retry_strategy = Retry(
    total=5,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/131 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/pdf,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://ntdc.gov.pk/"
})

## Download Angular JS Bundle

In [0]:
import re
from urllib.parse import urljoin

js_url = urljoin("https://ntdc.gov.pk/", "main.560d41594d04a9330a13.js")

print("Downloading:", js_url)

resp = session.get(
    js_url,
    timeout=60
)

print("Status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Size:", len(resp.content))

main_js = resp.text

print("\nDownloaded successfully:", len(main_js) > 0)

## Extract 2026 May–Aug Merit Order PDF Paths

In [0]:
import calendar
import re
from urllib.parse import quote

# Extract all PDF paths from the JavaScript bundle
path_pattern = r'/services/meritorder/[^"\)]+\.pdf'
all_paths = re.findall(path_pattern, main_js)
all_paths = list(dict.fromkeys(all_paths))  # Deduplicate

print(f"Total merit order PDF paths found: {len(all_paths)}")

# Map month numbers to full and abbreviated names for matching
month_names = {m: calendar.month_name[m] for m in TARGET_MONTHS}
month_abbr = {m: calendar.month_abbr[m] for m in TARGET_MONTHS}

print(f"\nFiltering for YEAR={TARGET_YEAR}, MONTHS={sorted(TARGET_MONTHS)}")
print(f"Target months: {[f'{month_names[m]} ({month_abbr[m]})' for m in sorted(TARGET_MONTHS)]}")

### Filter Paths by Year and Month

In [0]:
# Parse each path and filter by target year and months
filtered_paths = []

for path in all_paths:
    parts = path.split("/")
    # Expected format: ['', 'services', 'meritorder', '2026', 'July', 'EMO 28-07-2026.pdf']
    if len(parts) < 6:
        continue
    
    year_str = parts[3]
    month_folder = parts[4]
    filename = parts[5]
    
    # Validate year
    try:
        year = int(year_str)
    except ValueError:
        continue
    
    if year != TARGET_YEAR:
        continue
    
    # Match month folder against both full and abbreviated names
    for month_num in TARGET_MONTHS:
        full_name = month_names[month_num]
        abbr = month_abbr[month_num]
        
        if month_folder.lower() in (full_name.lower(), abbr.lower()):
            # Build the download URL with properly encoded filename
            encoded_filename = quote(filename, safe='/')
            download_url = f"https://ntdc.gov.pk/ntdc/uploads/services/meritorder/{TARGET_YEAR}/{month_folder}/{encoded_filename}"
            
            filtered_paths.append({
                "month_num": month_num,
                "month_name": full_name,
                "month_abbr": abbr,
                "month_folder": month_folder,
                "filename": filename,
                "js_path": path,
                "download_url": download_url
            })
            break

print(f"\nFiltered paths: {len(filtered_paths)} files found for {TARGET_YEAR}")

### Display Filtered PDFs

In [0]:
# Sort and display all filtered paths
print("=" * 100)
print(f"Found {len(filtered_paths)} PDFs across {len(TARGET_MONTHS)} months:\n")

for fp in sorted(filtered_paths, key=lambda x: (x["month_num"], x["filename"])):
    print(f"  [{fp['month_abbr']}] {fp['filename']}")
    print(f"    -> {fp['download_url']}")
    print()

# Build list of download URLs for downstream processing
download_urls = [fp["download_url"] for fp in sorted(filtered_paths, key=lambda x: (x["month_num"], x["filename"]))]

print("=" * 100)
print(f"Total download URLs prepared: {len(download_urls)}")

## Setting Up download Directories Upto Month Level

In [0]:
import os

# Setup base directory structure
base_volume_dir = f"{VOLUME_PATH}/meritorder/{TARGET_YEAR}"

print(f"Base directory: {base_volume_dir}")
print(f"Files to download: {len(filtered_paths)}")

# Create directories for each month
for fp in filtered_paths:
    month_folder = fp["month_folder"]
    target_dir = f"{base_volume_dir}/{month_folder}"
    dbutils.fs.mkdirs(target_dir)

print(f"Created directories for {len(set(fp['month_folder'] for fp in filtered_paths))} months")

## Download PDFs to Unity Catalog Volume

In [0]:
# Initialize tracking lists
downloaded = []
failed = []

print("=" * 80)
print("Starting download...\n")

for fp in sorted(filtered_paths, key=lambda x: (x["month_num"], x["filename"])):
    url = fp["download_url"]
    filename = fp["filename"]
    month_abbr = fp["month_abbr"]
    month_folder = fp["month_folder"]
    
    target_dir = f"{base_volume_dir}/{month_folder}"
    target_path = f"{target_dir}/{filename}"
    
    # Skip if file already exists
    try:
        existing = dbutils.fs.ls(target_dir)
        if any(f.name == filename for f in existing):
            print(f"[SKIP]  [{month_abbr}] {filename} (already exists)")
            downloaded.append(target_path)
            continue
    except Exception:
        pass
    
    # Download the file
    print(f"[GET]   [{month_abbr}] {filename}")
    try:
        resp = session.get(url, timeout=60)
        
        if resp.status_code == 200 and len(resp.content) > 0:
            # Write directly to volume
            with open(target_path, "wb") as f:
                f.write(resp.content)
            print(f"        -> Saved ({len(resp.content):,} bytes)")
            downloaded.append(target_path)
        else:
            print(f"        -> FAILED (status {resp.status_code})")
            failed.append(url)
    except Exception as e:
        print(f"        -> ERROR: {e}")
        failed.append(url)

print("\n" + "=" * 80)
print(f"Download complete: {len(downloaded)} successful, {len(failed)} failed")

### Verify Downloaded Files

In [0]:
# Display any failed downloads
if failed:
    print("Failed URLs:")
    for u in failed:
        print(f"  {u}")
    print()

# List all files in volume by month
print("Files in volume:")
print("=" * 80)

for month_folder in sorted(set(fp["month_folder"] for fp in filtered_paths)):
    month_dir = f"{base_volume_dir}/{month_folder}"
    try:
        files = dbutils.fs.ls(month_dir)
        print(f"\n  [{month_folder}] ({len(files)} files):")
        for f in sorted(files, key=lambda x: x.name):
            print(f"    {f.name} ({f.size:,} bytes)")
    except Exception as e:
        print(f"\n  [{month_folder}] Error: {e}")

### Extract Tables from PDFs using OCR

In [0]:
# Use ai_parse_document to OCR all PDFs and extract table HTML
extracted_df = spark.sql("""
    WITH parsed AS (
      SELECT
        _metadata.file_name AS file_name,
        ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
      FROM READ_FILES(
        '/Volumes/rf_assessment/bronze/rf_assessment_raw/meritorder/2026/',
        format => 'binaryFile'
      )
      WHERE _metadata.file_name LIKE '%.pdf'
    )
    SELECT
      file_name,
      transform(
        filter(
          try_cast(parsed_content:document:elements AS ARRAY<VARIANT>),
          e -> try_cast(e:type AS STRING) = 'table'
        ),
        e -> try_cast(e:content AS STRING)
      ) AS table_htmls
    FROM parsed
""")

print(f"Extracted table HTML from {extracted_df.count()} PDFs")
display(extracted_df.limit(3))

## Parse PDFs and Create Bronze Table

Uses Databricks `ai_parse_document` to OCR the image-based tables in each PDF, then parses the HTML with pandas to create structured data.

In [0]:
%pip install lxml -q

In [0]:
import pandas as pd
from io import StringIO
from datetime import datetime
from urllib.parse import unquote
import re

# Parse HTML tables with pandas and extract merit order data
rows = extracted_df.collect()
all_rows = []
expected_cols = ['sr_no', 'plant_name', 'fuel_type', 'other_cost',
                 'fuel_cost', 'vom_cost', 'specific_cost', 'status_last_order']

for row in rows:
    file_name = row['file_name']
    table_htmls = row['table_htmls']

    if table_htmls is None or len(table_htmls) == 0:
        print(f"[SKIP] No table found in {file_name}")
        continue

    # Try each table HTML and pick the one with 8 columns (merit order table)
    # Some PDFs have a small reference table before the main merit order table
    table_df = None
    for table_html in table_htmls:
        if table_html is None:
            continue
        try:
            tables = pd.read_html(StringIO(table_html))
            for t in tables:
                if len(t.columns) == len(expected_cols):
                    table_df = t
                    break
        except Exception:
            continue
        if table_df is not None:
            break

    if table_df is None:
        print(f"[SKIP] No 8-column table found in {file_name}")
        continue

    # Standardize column names by position (headers vary across PDFs)
    table_df.columns = expected_cols

    # Extract effective date from filename: "EMO 03-05-2026.pdf" -> 2026-05-03
    decoded_name = unquote(file_name)
    date_match = re.search(r'(\d{2})-(\d{2})-(\d{4})', decoded_name)
    if date_match:
        day, month, year = int(date_match.group(1)), int(date_match.group(2)), int(date_match.group(3))
        effective_date = f"{year}-{month:02d}-{day:02d}"
    else:
        print(f"[WARN] Could not extract date from {file_name}")
        effective_date, year, month = None, None, None

    # Add metadata columns
    table_df['file_name'] = decoded_name
    table_df['effective_date'] = effective_date
    table_df['year'] = year
    table_df['month'] = month
    table_df['ingestion_date'] = datetime.now()

    # Clean numeric columns: replace "-" and empty strings with 0
    for col in ['other_cost', 'fuel_cost', 'vom_cost', 'specific_cost']:
        table_df[col] = table_df[col].astype(str).str.strip()
        table_df[col] = table_df[col].replace(['-', '', 'nan', 'None'], '0')
        table_df[col] = pd.to_numeric(table_df[col], errors='coerce').fillna(0.0)

    # Convert integer columns
    table_df['sr_no'] = pd.to_numeric(table_df['sr_no'], errors='coerce').astype('Int64')
    table_df['status_last_order'] = pd.to_numeric(table_df['status_last_order'], errors='coerce').astype('Int64')

    all_rows.append(table_df)
    print(f"[OK] {decoded_name}: {len(table_df)} rows")

print("\n" + "=" * 80)
print(f"Parsing complete: {len(all_rows)} PDFs processed successfully")

### Combine Data and Write to Bronze Table

In [0]:
if not all_rows:
    print("No data extracted!")
else:
    # Combine all dataframes
    final_df = pd.concat(all_rows, ignore_index=True)
    
    print(f"Total rows across all PDFs: {len(final_df)}")
    print(f"Columns: {list(final_df.columns)}")
    print(f"\nRows per month:")
    print(final_df.groupby(['year', 'month']).size().to_string())
    
    # Convert to Spark DataFrame with explicit schema
    from pyspark.sql.types import (
        StructType, StructField, IntegerType, StringType,
        DoubleType, TimestampType
    )
    
    schema = StructType([
        StructField("sr_no", IntegerType(), True),
        StructField("plant_name", StringType(), True),
        StructField("fuel_type", StringType(), True),
        StructField("other_cost", DoubleType(), True),
        StructField("fuel_cost", DoubleType(), True),
        StructField("vom_cost", DoubleType(), True),
        StructField("specific_cost", DoubleType(), True),
        StructField("status_last_order", IntegerType(), True),
        StructField("file_name", StringType(), True),
        StructField("effective_date", StringType(), True),
        StructField("year", IntegerType(), True),
        StructField("month", IntegerType(), True),
        StructField("ingestion_date", TimestampType(), True),
    ])
    
    # Convert pandas NaN to None for Spark compatibility
    final_df = final_df.where(pd.notnull(final_df), None)
    final_df = final_df.astype({
        'sr_no': 'float',
        'status_last_order': 'float',
        'year': 'float',
        'month': 'float',
    })
    
    spark_df = spark.createDataFrame(final_df, schema=schema)
    
    # Write to bronze table
    spark_df.write \
        .mode("overwrite") \
        .saveAsTable("rf_assessment.bronze.merit_order_data")
    
    print("\n" + "=" * 80)
    print("Bronze table created: rf_assessment.bronze.merit_order_data")
    print(f"Total rows: {spark_df.count()}")